## RQ1: Platform-Mediated Social Diagnosis
#### Research Question: How can online platform data serve as a diagnostic tool for identifying systemic social friction and integration patterns in diverse societies?

In [ ]:
import pandas as pd

In [ ]:
### Reading the merged dataframe for analys
analysis_df = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/final_merged_data_for_analysis_03122025.parquet")
analysis_df.info()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Professional color palette that works for all plot types
PRO_PALETTE = {
    'main': ['#2E5EAA', '#C44E52', '#55A868', '#8172B3', '#CCB974', 
             '#64B5CD', '#8C564B', '#E377C2', '#7F7F7F', '#BCBD22'],
    'sequential': ['#f7fbff', '#deebf7', '#c6dbef', '#9ecae1', '#6baed6', 
                   '#4292c6', '#2171b5', '#08519c', '#08306b'],  # Blues
    'diverging': ['#b2182b', '#d6604d', '#f4a582', '#fddbc7', '#f7f7f7', 
                  '#d1e5f0', '#92c5de', '#4393c3', '#2166ac'],  # Red-Blue
    'heatmap': 'viridis'  # Good default for heatmaps
}

plt.style.use('default')  # Reset to default for better control
sns.set_theme(style="whitegrid")  # Use seaborn's modern theme

# Set consistent style parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'DejaVu Sans'  # Good academic font

# Professional styling
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['grid.color'] = '#E5E5E5'
plt.rcParams['grid.alpha'] = 0.7
plt.rcParams['axes.edgecolor'] = '#2E2E2E'
plt.rcParams['axes.labelcolor'] = '#2E2E2E'
plt.rcParams['text.color'] = '#2E2E2E'
plt.rcParams['xtick.color'] = '#2E2E2E'
plt.rcParams['ytick.color'] = '#2E2E2E'

# Set the color cycle for plots
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=PRO_PALETTE['main'])

### Step 1: Prevalence Analysis

In [ ]:
print("="*60)
print("RQ1: PLATFORM-MEDIATED SOCIAL DIAGNOSIS")
print("="*60)

# Create a numeric flag for racialized reviews (1 = yes, 0 = no) to avoid string arithmetic
analysis_df['racialized_flag'] = (analysis_df['racial_connotations'].astype(str).str.lower() == 'yes').astype(int)

# Calculate overall prevalence using the numeric flag
racialized_count = int(analysis_df['racialized_flag'].sum())
total_reviews = len(analysis_df)
racialized_rate = racialized_count / total_reviews if total_reviews > 0 else 0.0

print(f"\n1. OVERALL PREVALENCE:")
print(f"   • {racialized_count:,} racialized reviews out of {total_reviews:,} total")
print(f"   • Overall rate: {racialized_rate:.2%}")

In [ ]:
# Prepare numeric columns for aggregation (coerce errors to NaN)
analysis_df['reviewer_rating_num'] = pd.to_numeric(analysis_df['reviewer_rating'], errors='coerce')
analysis_df['uploaded_pics_num'] = pd.to_numeric(analysis_df['uploaded_pics'], errors='coerce')
analysis_df['other_reviews_num'] = pd.to_numeric(analysis_df['other_reviews'], errors='coerce')

# Platform engagement patterns: group by the numeric flag
platform_analysis = analysis_df.groupby('racialized_flag').agg({
    'reviewer_rating_num': ['mean', 'count'],
    'uploaded_pics_num': 'mean',
    'other_reviews_num': 'mean'
}).round(3)

# Safely retrieve values (handle missing groups)
rating_series = platform_analysis[('reviewer_rating_num', 'mean')] if ('reviewer_rating_num', 'mean') in platform_analysis.columns else pd.Series(dtype=float)
uploaded_series = platform_analysis[('uploaded_pics_num', 'mean')] if ('uploaded_pics_num', 'mean') in platform_analysis.columns else pd.Series(dtype=float)

rating_racialized = rating_series.get(1, np.nan)
rating_nonracialized = rating_series.get(0, np.nan)
pics_racialized = uploaded_series.get(1, np.nan)
pics_nonracialized = uploaded_series.get(0, np.nan)

print(f"\n2. PLATFORM ENGAGEMENT PATTERNS:")
if not np.isnan(rating_racialized) and not np.isnan(rating_nonracialized):
    print(f"   • Racialized reviews have lower average ratings ({rating_racialized:.2f} vs {rating_nonracialized:.2f})")
else:
    print("   • Rating summary not available for both groups (missing data)")

if not np.isnan(pics_racialized) and not np.isnan(pics_nonracialized):
    print(f"   • Photo upload rates: racialized={pics_racialized:.3f}, non-racialized={pics_nonracialized:.3f}")
else:
    print("   • Photo upload summary not available for both groups (missing data)")



In [ ]:
# State-level analysis
# Use left join so we keep reviews even if a pool match is missing; then compute mean of the flag per state

state_rates = (analysis_df.dropna(subset=['land_name'])
                           .groupby('land_name')['racialized_flag']
                           .mean()
                           .sort_values(ascending=False))

print(f"\n3. GEOGRAPHIC VARIATION:")
if state_rates.empty:
    # If still empty, inform the user and give a hint
    print("   • No geographic matches found between reviews and pools (land_name missing).")
    print("     Check that 'place_id' values match in both dataframes or that pools_df[['place_id', 'land_name']] contains data.")
else:
    # Print top/bottom, handling cases with fewer than 4 states
    n = len(state_rates)
    if n >= 2:
        top_n = min(2, n)
        bottom_n = min(2, n)
        top_items = state_rates.head(top_n)
        bottom_items = state_rates.tail(bottom_n)
        top_str = ", ".join([f"{idx} ({val:.2%})" for idx, val in top_items.items()])
        bottom_str = ", ".join([f"{idx} ({val:.2%})" for idx, val in bottom_items.items()][::-1])
        print(f"   • Highest rates: {top_str}")
        print(f"   • Lowest rates: {bottom_str}")
    else:
        # Only one state present
        idx = state_rates.index[0]
        val = state_rates.iloc[0]
        print(f"   • Only one state with data: {idx} ({val:.2%})")

### Step 2: Temporal Analysis

In [ ]:
# Temporal trends (robust parsing for relative strings like "2 years ago", "Edited 7 years ago", "a year ago")
import re
from datetime import datetime

def parse_review_time(s, ref=None):
    """Try to parse absolute dates first; if that fails, handle relative strings like '2 years ago'."""
    if pd.isna(s):
        return pd.NaT
    if ref is None:
        ref = pd.Timestamp.today()
    s2 = str(s).strip()
    # remove common prefixes like 'Edited '
    s2 = re.sub(r'^[Ee]dited\s+', '', s2)
    # try absolute parse
    dt = pd.to_datetime(s2, errors='coerce', dayfirst=True)
    if not pd.isna(dt):
        return dt
    # relative patterns
    m = re.match(r'^(?:about\s+)?(\d+)\s+years?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.DateOffset(years=int(m.group(1)))
    if re.match(r'^(?:about\s+)?a\s+year\s+ago$', s2, flags=re.I):
        return ref - pd.DateOffset(years=1)
    m = re.match(r'^(?:about\s+)?(\d+)\s+months?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.DateOffset(months=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+weeks?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(weeks=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+days?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(days=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+hours?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(hours=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+minutes?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(minutes=int(m.group(1)))
    # fallback
    return pd.NaT

In [ ]:


# Apply parsing (creates a new parsed datetime column)
analysis_df['review_time_parsed'] = analysis_df['review_time'].apply(parse_review_time)
# Try a final coercion for any remaining unparsed values
analysis_df['review_time_parsed'] = analysis_df['review_time_parsed'].fillna(
    pd.to_datetime(analysis_df['review_time'], errors='coerce', dayfirst=True)
)

# Derive year from parsed datetime
analysis_df['year'] = analysis_df['review_time_parsed'].dt.year

# Use existing numeric flag column 'racialized_flag' (not 'is_racialized')
temporal_trends = analysis_df.groupby('year')['racialized_flag'].mean().dropna().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: State variation (use state_rates if available)
state_rates_sorted = state_rates.sort_values() if 'state_rates' in globals() else pd.Series(dtype=float)
if not state_rates_sorted.empty:
    bars = ax1.barh(range(len(state_rates_sorted)), state_rates_sorted.values,
                    color=PRO_PALETTE['main'][0], alpha=0.7)
    ax1.set_yticks(range(len(state_rates_sorted)))
    ax1.set_yticklabels(state_rates_sorted.index)
    ax1.set_xlabel('Proportion of Racialized Reviews')
    ax1.set_title('A. Geographic Variation in Racialized Discourse', pad=20)
    ax1.grid(axis='x', alpha=0.3)
    # Add value annotations
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax1.text(width + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{width:.2%}', ha='left', va='center', fontsize=9)
else:
    ax1.text(0.5, 0.5, 'No geographic rates available', ha='center', va='center')
    ax1.set_axis_off()

# Plot 2: Temporal trends
if not temporal_trends.empty:
    ax2.plot(temporal_trends.index.astype(int), temporal_trends.values,
             marker='o', linewidth=2.5, markersize=8, color=PRO_PALETTE['main'][1])
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Proportion of Racialized Reviews')
    ax2.set_title('B. Temporal Trends in Racialized Discourse', pad=20)
    ax2.grid(True, alpha=0.3)
    ymax = temporal_trends.max() * 1.1 if temporal_trends.max() > 0 else 0.1
    ax2.set_ylim(0, ymax)
    # Add value annotations (only for valid numeric years)
    for year, rate in temporal_trends.items():
        if not pd.isna(rate):
            ax2.text(int(year), rate + (ymax * 0.02), f'{rate:.2%}', ha='center', va='bottom', fontsize=9)
else:
    ax2.text(0.5, 0.5, 'No temporal data available', ha='center', va='center')
    ax2.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Temporal trends (robust parsing for relative strings like "2 years ago", "Edited 7 years ago", "a year ago")
import re
from datetime import datetime

def parse_review_time(s, ref=None):
    """Try to parse absolute dates first; if that fails, handle relative strings like '2 years ago'."""
    if pd.isna(s):
        return pd.NaT
    if ref is None:
        ref = pd.Timestamp.today()
    s2 = str(s).strip()
    # remove common prefixes like 'Edited '
    s2 = re.sub(r'^[Ee]dited\s+', '', s2)
    # try absolute parse
    dt = pd.to_datetime(s2, errors='coerce', dayfirst=True)
    if not pd.isna(dt):
        return dt
    # relative patterns
    m = re.match(r'^(?:about\s+)?(\d+)\s+years?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.DateOffset(years=int(m.group(1)))
    if re.match(r'^(?:about\s+)?a\s+year\s+ago$', s2, flags=re.I):
        return ref - pd.DateOffset(years=1)
    m = re.match(r'^(?:about\s+)?(\d+)\s+months?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.DateOffset(months=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+weeks?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(weeks=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+days?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(days=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+hours?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(hours=int(m.group(1)))
    m = re.match(r'^(?:about\s+)?(\d+)\s+minutes?\s+ago$', s2, flags=re.I)
    if m:
        return ref - pd.Timedelta(minutes=int(m.group(1)))
    # fallback
    return pd.NaT

# Apply parsing (creates a new parsed datetime column)
analysis_df['review_time_parsed'] = analysis_df['review_time'].apply(parse_review_time)
# Try a final coercion for any remaining unparsed values
analysis_df['review_time_parsed'] = analysis_df['review_time_parsed'].fillna(
    pd.to_datetime(analysis_df['review_time'], errors='coerce', dayfirst=True)
)

# Derive year from parsed datetime
analysis_df['year'] = analysis_df['review_time_parsed'].dt.year

# Use existing numeric flag column 'racialized_flag' (not 'is_racialized')
temporal_trends = analysis_df.groupby('year')['racialized_flag'].mean().dropna().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Plot 1: Stacked bar chart for land_name variation with connotation types
# Filter out 'MISSING' connotation type
filtered_df = analysis_df[analysis_df['connotation_type'].isin(['pleasant', 'discriminatory', 'ambiguous'])]

# Calculate the breakdown by land_name and connotation type
state_connotation_counts = filtered_df.groupby(['land_name', 'connotation_type']).size().unstack(fill_value=0)

# Calculate the percentage of each connotation type within each land_name
state_connotation_percent = state_connotation_counts.div(state_connotation_counts.sum(axis=1), axis=0)

# Sort states by total racialized reviews (if you want, or by something else)
state_totals = state_connotation_counts.sum(axis=1)
state_connotation_percent_sorted = state_connotation_percent.loc[state_totals.sort_values(ascending=False).index]

if not state_connotation_percent_sorted.empty:
    # Define colors for different connotation types
    connotation_colors = {
        'pleasant': '#d62728',    # Red
        'discriminatory': '#2ca02c',    # Green
        'ambiguous': '#ff7f0e'    # Orange
    }
    
    # Get the list of connotation types (excluding any that might be missing from the filtered data)
    connotation_types = [ct for ct in connotation_colors.keys() if ct in state_connotation_percent_sorted.columns]
    
    # Create stacked bars
    bottom_values = pd.Series(0, index=state_connotation_percent_sorted.index)
    
    for connotation in connotation_types:
        if connotation in state_connotation_percent_sorted.columns:
            ax1.barh(
                range(len(state_connotation_percent_sorted)), 
                state_connotation_percent_sorted[connotation] * 100,  # Convert to percentage
                left=bottom_values,
                color=connotation_colors[connotation],
                label=f'{connotation}',
                alpha=0.8,
                edgecolor='white',
                linewidth=0.5
            )
            bottom_values += state_connotation_percent_sorted[connotation] * 100
    
    ax1.set_yticks(range(len(state_connotation_percent_sorted)))
    ax1.set_yticklabels(state_connotation_percent_sorted.index)
    ax1.set_xlabel('Percentage of Connotation Types in Racialized Reviews (%)', fontsize=12)
    ax1.set_title('A. Geographic Distribution of Racialized Discourse by Connotation Type', pad=20, fontsize=14)
    ax1.grid(axis='x', alpha=0.3, linestyle='--')
    
    # Add legend
    ax1.legend(title='Connotation Type', bbox_to_anchor=(1.05, 1), loc='upper left', framealpha=0.9)
    
    # Add value annotations for the total percentage (optional)
    for i, land_name in enumerate(state_connotation_percent_sorted.index):
        total_pct = state_connotation_percent_sorted.loc[land_name].sum() * 100
        ax1.text(total_pct + 1, i, f'{total_pct:.0f}%', va='center', fontsize=9, 
                fontweight='bold', color='darkblue')
    
    # Set xlim to accommodate annotations
    ax1.set_xlim(0, state_connotation_percent_sorted.sum(axis=1).max() * 100 + 15)
    
    # Add a note about what's being shown
    ax1.text(0.02, 0.98, 'Note: Percentages show distribution of connotation types\nwithin racialized reviews for each land_name',
             transform=ax1.transAxes, fontsize=9, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
else:
    ax1.text(0.5, 0.5, 'No connotation type data available (excluding "MISSING")', 
             ha='center', va='center', fontsize=12)
    ax1.set_axis_off()

# Plot 2: Temporal trends (unchanged)
if not temporal_trends.empty:
    ax2.plot(temporal_trends.index.astype(int), temporal_trends.values * 100,  # Convert to percentage
             marker='o', linewidth=2.5, markersize=8, color=PRO_PALETTE['main'][1])
    ax2.set_xlabel('Year', fontsize=12)
    ax2.set_ylabel('Percentage of Racialized Reviews (%)', fontsize=12)
    ax2.set_title('B. Temporal Trends in Racialized Discourse', pad=20, fontsize=14)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ymax = temporal_trends.max() * 110  # Add 10% buffer for annotations
    ax2.set_ylim(0, ymax)
    # Add value annotations (only for valid numeric years)
    for year, rate in temporal_trends.items():
        if not pd.isna(rate):
            ax2.text(int(year), rate * 100 + (ymax * 0.02), f'{rate*100:.1f}%', 
                     ha='center', va='bottom', fontsize=9, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'No temporal data available', ha='center', va='center', fontsize=12)
    ax2.set_axis_off()

plt.tight_layout()
plt.show()

# Optional: Print summary statistics
print("Connotation Type Distribution Summary:")
print("=" * 50)
if not state_connotation_percent_sorted.empty:
    overall_connotation_dist = state_connotation_counts.sum().sort_values(ascending=False)
    overall_connotation_percent = (overall_connotation_dist / overall_connotation_dist.sum() * 100).round(1)
    
    for connotation, pct in overall_connotation_percent.items():
        count = overall_connotation_dist[connotation]
        print(f"{connotation}: {pct}% ({count} reviews)")
    
    print(f"\nTotal racialized reviews analyzed: {overall_connotation_dist.sum()}")
    print(f"Number of states with data: {len(state_connotation_percent_sorted)}")